## Embedding Caching
### 1. Import Required Libraries

Import the required libraries for generating cache keys, serializing embeddings, measuring execution time, and comparing embedding vectors.

In [2]:
import hashlib
import json
import time
import numpy as np
import redis
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:

redis_client = redis.Redis(
    host="localhost",
    port=6380,
    decode_responses=True
)

print("Redis connected:", redis_client.ping())
print("Redis version:", redis_client.info("server")["redis_version"])

Redis connected: True
Redis version: 8.10.3


### 2. Implement RedisEmbeddingCache

Create a reusable caching class that checks Redis before calling the embedding model.

**Workflow:**
- Generate a unique SHA-256 key from the input text.
- Check whether the embedding exists in Redis.
- Cache hit: Retrieve the stored embedding.
- Cache miss: Generate the embedding and store it in Redis.
- Apply TTL to automatically expire cached embeddings.

**Why:** Avoid redundant embedding generation and reduce CPU utilization and latency.

In [4]:
class RedisEmbeddingCache:
    def __init__(self, redis_client, embedding_model, ttl=86400):
        self.redis = redis_client
        self.model = embedding_model
        self.ttl = ttl
        self.hits = 0
        self.misses = 0

    def get_embedding(self, text: str):
        cache_key = hashlib.sha256(text.encode()).hexdigest()
        key = f"seva:embedding:v1:{cache_key}"

        cached = self.redis.get(key)

        if cached is not None:
            self.hits += 1
            return json.loads(cached)

        self.misses += 1
        embedding = self.model.embed_query(text)

        self.redis.set(key, json.dumps(embedding), ex=self.ttl)

        return embedding

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


### 3. Initialize Redis Embedding Cache

Initialize the caching class using the existing Redis connection and Hugging Face embedding model.

**Parameters:**
- `redis_client`: Existing Redis connection.
- `embeddings`: Existing embedding model.
- `ttl=86400`: Cache expiration time of 24 hours.

In [7]:
embedding_cache = RedisEmbeddingCache(
    redis_client=redis_client,
    embedding_model=embedding_model,
    ttl=86400
)

### 4. Test Cache Hit and Miss

Execute the same query twice.

**Expected behaviour:**
- First request: Cache miss; generate and store embedding.
- Second request: Cache hit; retrieve embedding from Redis.

Measure the latency of both requests to understand the performance improvement.

In [8]:
query = "Which government schemes provide financial assistance to farmers?"

# First request: Cache miss
start = time.perf_counter()
embedding_1 = embedding_cache.get_embedding(query)
first_latency = time.perf_counter() - start

# Second request: Cache hit
start = time.perf_counter()
embedding_2 = embedding_cache.get_embedding(query)
second_latency = time.perf_counter() - start

print("First request:", round(first_latency * 1000, 2), "ms")
print("Second request:", round(second_latency * 1000, 2), "ms")

print("Embedding dimensions:", len(embedding_1))
print("Embeddings identical:", np.allclose(embedding_1, embedding_2))
print("Cache hits:", embedding_cache.hits)
print("Cache misses:", embedding_cache.misses)

First request: 1470.54 ms
Second request: 15.43 ms
Embedding dimensions: 384
Embeddings identical: True
Cache hits: 1
Cache misses: 1


### 5. Calculate Cache Hit Rate and Miss Rate

Calculate the percentage of requests served from cache versus requests requiring embedding generation.

**Formulas:**

Hit Rate = (Cache Hits / Total Requests) × 100

Miss Rate = (Cache Misses / Total Requests) × 100

A higher hit rate generally indicates better cache reuse.

In [9]:
total_requests = embedding_cache.hits + embedding_cache.misses

hit_rate = embedding_cache.hits / total_requests * 100 if total_requests else 0
miss_rate = embedding_cache.misses / total_requests * 100 if total_requests else 0

print(f"Cache Hit Rate: {hit_rate:.2f}%")
print(f"Cache Miss Rate: {miss_rate:.2f}%")

Cache Hit Rate: 50.00%
Cache Miss Rate: 50.00%


### 6. Experiment Observations

Record and analyze the following metrics:

- First request latency (cache miss).
- Second request latency (cache hit).
- Cache hit rate.
- Cache miss rate.
- Embedding dimensions.
- Redis memory consumption.

**Expected result for two identical requests:**
- Cache hits: 1
- Cache misses: 1
- Hit rate: 50%
- Miss rate: 50%
- Embedding dimensions: 384

### Conclusion

Redis embedding caching avoids repeatedly generating embeddings for identical queries. This can reduce embedding-generation latency and CPU utilization.

**Future improvements:**
- Model-version-aware cache keys.
- Query normalization.
- Binary serialization.
- Cache invalidation.
- Cache stampede prevention.
- TTL and eviction policy optimization.

### Exact Query Caching

Implement an exact query cache using Redis to store and retrieve previously computed retrieval results.

**Why do we need it?**

When users submit the same query repeatedly, the RAG system performs vector retrieval each time.

Instead, we can cache the retrieval results and reuse them for identical queries.

**Expected benefits:**
- Reduce repeated vector-search operations.
- Reduce retrieval latency.
- Reduce Chroma database workload.
- Improve response time for frequently repeated queries.

**Difference from embedding caching:**
- Embedding cache stores query vectors.
- Query cache stores retrieval results, such as retrieved chunks, metadata, and scores.

### 2. Implement RedisQueryCache

Create a reusable class to cache retrieval results.

**Workflow:**
1. Receive a user query.
2. Generate a unique cache key.
3. Check Redis for existing results.
4. Cache hit: Return previously stored results.
5. Cache miss: Execute the retrieval function.
6. Serialize and store the results in Redis.
7. Apply TTL for automatic expiration.

The retrieval function will be passed as a callback so the cache can work with different retrieval pipelines.

In [10]:
import hashlib
import json

class RedisQueryCache:
    def __init__(self, redis_client, ttl=3600):
        self.redis = redis_client
        self.ttl = ttl
        self.hits = 0
        self.misses = 0

    def get_or_compute(self, query, retrieval_fn):
        cache_key = hashlib.sha256(query.encode()).hexdigest()
        key = f"seva:retrieval:v1:{cache_key}"

        cached = self.redis.get(key)

        if cached is not None:
            self.hits += 1
            return json.loads(cached)

        self.misses += 1

        results = retrieval_fn(query)

        serialized_results = json.dumps(results)

        self.redis.set(key, serialized_results, ex=self.ttl)

        return results

### 3. Initialize Redis Query Cache

Initialize the query cache with the existing Redis connection.

**Configuration:**
- Redis client: Existing `redis_client`.
- TTL: 3600 seconds (1 hour).
- Cache namespace: `seva:retrieval:v1`.

The one-hour TTL is an experimental default, not a final production policy.

In [11]:
query_cache = RedisQueryCache(
    redis_client=redis_client,
    ttl=3600
)

### 4. Create Baseline Retrieval Function

Use the existing Chroma discovery index to retrieve the top 10 matching chunks.

Convert the retrieved documents into dictionaries so they can be serialized and stored in Redis.

**Note:** This is a baseline retrieval function for the caching experiment. We will integrate the complete discovery and detail retrieval pipeline later.

In [14]:
from pathlib import Path
from collections import defaultdict
from langchain_chroma import Chroma

OUTPUT_DIR1 = Path("G:\\SEVA_Ai\\SEVA_Ai\\output\\SEVA_CHILDREN")

VECTORSTORE_DIR = OUTPUT_DIR1 / "vectorstores"

PARENT_PATH = OUTPUT_DIR1 / "parent_documents.jsonl"

print("Parent file exists:", PARENT_PATH.exists())
print("Vectorstore directory exists:", VECTORSTORE_DIR.exists())

Parent file exists: True
Vectorstore directory exists: True


In [15]:
discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [16]:
def retrieve_schemes(query):
    results = discovery_db.similarity_search_with_score(
        query,
        k=10
    )

    return [
        {
            "page_content": doc.page_content,
            "metadata": doc.metadata,
            "distance": float(score)
        }
        for doc, score in results
    ]

### 5. Test Cache Hit and Miss

Execute the same query twice.

**Expected behaviour:**
- First request: Cache miss; execute Chroma retrieval and cache the results.
- Second request: Cache hit; retrieve the stored results directly from Redis.

Measure both execution times to compare retrieval latency.

In [17]:
query = "Which government schemes provide financial assistance to farmers?"

# First request: Cache miss
start = time.perf_counter()
results_1 = query_cache.get_or_compute(query, retrieve_schemes)
first_latency = time.perf_counter() - start

# Second request: Cache hit
start = time.perf_counter()
results_2 = query_cache.get_or_compute(query, retrieve_schemes)
second_latency = time.perf_counter() - start

print("First request:", round(first_latency * 1000, 2), "ms")
print("Second request:", round(second_latency * 1000, 2), "ms")

print("Retrieved documents:", len(results_1))
print("Results identical:", results_1 == results_2)

print("Cache hits:", query_cache.hits)
print("Cache misses:", query_cache.misses)

First request: 940.31 ms
Second request: 21.0 ms
Retrieved documents: 10
Results identical: True
Cache hits: 1
Cache misses: 1


### 6. Calculate Cache Hit Rate and Miss Rate

Measure the percentage of retrieval requests served directly from Redis.

**Hit Rate** = (Cache Hits / Total Requests) × 100

**Miss Rate** = (Cache Misses / Total Requests) × 100

In [18]:
total_requests = query_cache.hits + query_cache.misses

hit_rate = query_cache.hits / total_requests * 100 if total_requests else 0
miss_rate = query_cache.misses / total_requests * 100 if total_requests else 0

print(f"Cache Hit Rate: {hit_rate:.2f}%")
print(f"Cache Miss Rate: {miss_rate:.2f}%")

Cache Hit Rate: 50.00%
Cache Miss Rate: 50.00%


### 7. Experiment Observations

Record:

- First retrieval latency.
- Second retrieval latency.
- Cache hit rate.
- Cache miss rate.
- Number of retrieved documents.
- Redis memory consumption.

**Expected result for two identical requests:**
- Cache hits: 1
- Cache misses: 1
- Hit rate: 50%
- Miss rate: 50%

### Production Considerations

1. Include embedding model and index version in cache keys.
2. Normalize queries consistently.
3. Invalidate cached results when the underlying index changes.
4. Apply appropriate TTL.
5. Prevent cache stampedes during concurrent requests.
6. Avoid caching sensitive user-specific results without proper isolation.

### Conclusion

Exact query caching avoids repeated vector retrieval for identical queries and reduces unnecessary database operations.